# **Synthetic Data Generation Models with Utility and Privacy Metrics Comparisons for AI/ML Project WP13**

# **System Preparation for Clouds and All Installations: Azure**

In [ ]:
import os
import platform

def is_running_on_azure():
    return "AML_CloudName" in os.environ

def is_running_onyxia(): 
    return None

if is_running_on_azure() is True:
    !sudo apt-get update
    !sudo apt-get install -y p7zip-full

    %pip install azure-identity==1.12.0
    %pip install --upgrade gdown
    %pip install matplotlib
    %pip install seaborn
    %pip uninstall -y numpy
    %pip install --no-cache-dir --force-reinstall numpy
    %pip install --no-cache-dir --force-reinstall scipy pandas scikit-learn
    %pip install openpyxl
    %pip install imblearn
%pip install --upgrade pip
%pip install --upgrade --no-cache-dir sdv
%pip install category_encoders
%pip install python-docx
%pip install smartnoise-synth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 277.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 492.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [sdv]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 30.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [smartnoise-synth]


# **All Imports**

In [ ]:
# General Libraries
import pickle
import timeit
import random
import pandas as pd
import numpy as np
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from IPython import get_ipython
import category_encoders as ce
from collections import Counter

# Machine Learning Libraries
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBClassifier, XGBRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, recall_score, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.tree import plot_tree
import missingno as msno
from sklearn.utils import class_weight

if is_running_on_azure() is False:
    import tensorflow as tf
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

# Generative Models
from sdv.single_table import TVAESynthesizer, CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
from imblearn.over_sampling import SMOTENC, SMOTE
from snsynth import Synthesizer

from sdmetrics.single_table.privacy import (
    DisclosureProtection,
    DisclosureProtectionEstimate,
    DCROverfittingProtection,
    DCRBaselineProtection
)

# **Globals**

In [ ]:
training = False
RF = True
XGB = True
SMOTE = True
VAE = True
CTGAN = True
DPCTGAN = True

seed = 42
base_dir = "AIML4OS_WP_13_synthetic_data_modelling_task_team"
if is_running_on_azure() is True:
    local_path = "../Dataset/PUFmerged.csv"
else:
    local_path = "./"+base_dir+"/Dataset/PUFmerged.csv"
dataset_limit = None                # None: takes all the samples

# selected features to feed the model
features = []

# all features to feed the model
'''
all_features_old = ["tr","mese","annrif","sesso","eta10","staciv4",
                "pnasc","reg","rip","istr4","condogg","posiz4","ateco3","cond4","ORARIO",
                "RAPP","CODPROF7","progvia","TIPOVGG","DEST_RE","DEST_PR","DEST_IE",
                "GGINIZ","MMINIZ","AAINIZ","DURATA","MEZZO","ALLOG",
                "ORGALL","ORGTRA","ALPART","IORGALL","IORGTRA","TIPOMARE","TIPOCROC",
                "TIPOMONT","TIPOCITTA","TIPOCAMP","TIPOALTRO","W","COEV","TRIM","npart",
                "tipo","ESPE_CO","ESPE_GIO","piattall1","piattall2","piattall3"]
'''
# featues selected with Foschi's analysis

'''
all_features = ["sesso","eta10","staciv4",
                "pnasc","reg","rip","istr4","condogg","posiz4","ateco3","cond4","ORARIO",
                "RAPP","CODPROF7","progvia","TIPOVGG","DEST_RE","DEST_PR","DEST_IE",
                "GGINIZ","MMINIZ","AAINIZ","DURATA","MEZZO","ALLOG",
                "ORGALL","ORGTRA","ALPART","IORGALL","IORGTRA","TIPOMARE","TIPOCROC",
                "TIPOMONT","TIPOCITTA","TIPOCAMP","TIPOALTRO","npart",
                "tipo","ESPE_CO","ESPE_GIO","piattall1","piattall2","piattall3"]
'''

# we made a cut up to 0.2 for the two principal components
'''
all_features = ["TIPOMARE", "TIPOCROC", "TIPOMONT", "TIPOCITTA", "TIPOCAMP","TIPOALTRO", "tipo", "piattall1", "piattall2", "piattall3", "condogg", "cond4"]
'''

features = ["tr","mese","annrif","progind", "sesso","eta10","staciv4",
                "pnasc","reg","rip","istr4","condogg","posiz4","ateco3","cond4","ORARIO",
                "RAPP","CODPROF7","progvia","DEST_RE","DEST_PR","DEST_IE",
                "GGINIZ","MMINIZ","AAINIZ","DURATA","MEZZO","ALLOG",
                "ORGALL","ORGTRA","ALPART","IORGALL","IORGTRA","TIPOMARE","TIPOCROC",
                "TIPOMONT","TIPOCITTA","TIPOCAMP","TIPOALTRO","W","COEV","TRIM","npart",
                "tipo","ESPE_CO","ESPE_GIO","piattall1","piattall2","piattall3"]

# Target Variable
target = "MOTVAC"

save_model = False

# Privacy Metrics
dp_privacy = True
dcr_privacy = True
mia_privacy = True
n_dp_limit = None
n_dcr_limit = 2000
n_mia_limit = None

# **Hyper-Parameters**

In [ ]:
# Differential Privacy CTGAN
dpctgan_epsilon = 3.0   # moderate privacy
dpctgan_delta = 1e-5   # High privacy guarantee
dpctgan_epochs = 100
dpctgan_batch_size = 2048

# **Initializations**

In [ ]:
start_global_time = timeit.default_timer()

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 100)

# for reproducibility
random.seed(seed)
np.random.seed(seed)

local_results_path = f"./{base_dir}/Results/Synthetic_Data/"

# **Functions Definition**

In [ ]:
def match_datatypes(original_df, synthetic_df):
    """
    Matches the data types of a synthetic DataFrame to an original DataFrame.

    Args:
        original_df (pd.DataFrame): The original DataFrame whose dtypes are to be matched.
        synthetic_df (pd.DataFrame): The synthetic DataFrame to be converted.

    Returns:
        pd.DataFrame: The synthetic DataFrame with data types matched to the original.
    """
    original_dtypes = original_df.dtypes
    processed_synthetic_df = synthetic_df.copy()

    for col in processed_synthetic_df.columns:
        if col in original_dtypes.index:
            target_dtype = original_dtypes[col]

            if pd.api.types.is_numeric_dtype(target_dtype):
                if pd.api.types.is_integer_dtype(target_dtype):
                    processed_synthetic_df[col] = processed_synthetic_df[col].round().astype(target_dtype)
                else:
                    processed_synthetic_df[col] = processed_synthetic_df[col].astype(target_dtype)
            elif pd.api.types.is_object_dtype(target_dtype):
                processed_synthetic_df[col] = processed_synthetic_df[col].astype(str)
            else:
                try:
                    processed_synthetic_df[col] = processed_synthetic_df[col].astype(target_dtype)
                except Exception as e:
                    print(f"Warning: Could not convert column '{col}' to original type '{target_dtype}': {e}")
        else:
            print(f"Column '{col}' not found in original df. Type conversion skipped for this column.")
    return processed_synthetic_df

def set_table_borders(table):
    tbl = table._tbl
    tblPr = tbl.tblPr

    borders = OxmlElement('w:tblBorders')

    for border_name in ['top', 'left', 'bottom', 'right', 'insideH', 'insideV']:
        border = OxmlElement(f'w:{border_name}')
        border.set(qn('w:val'), 'single')
        border.set(qn('w:sz'), '8')
        border.set(qn('w:space'), '0')
        border.set(qn('w:color'), '000000')
        borders.append(border)

    tblPr.append(borders)

from docx import Document

def save_results_to_word(results_dict, output_path, title, score_label="Score"):

    doc = Document()
    doc.add_heading(title, level=1)

    first_value = next(iter(results_dict.values()))

    # ---- DCR / Disclosure Protection
    if not isinstance(first_value, dict):

        table = doc.add_table(rows=1, cols=2)
        set_table_borders(table)

        headers = ["Method", score_label]

        for i, h in enumerate(headers):
            run = table.rows[0].cells[i].paragraphs[0].add_run(h)
            run.bold = True

        for method, score in results_dict.items():
            row = table.add_row().cells
            row[0].text = str(method)
            row[1].text = f"{float(score):.4f}"

    # ---- MIA
    else:

        table = doc.add_table(rows=1, cols=5)
        set_table_borders(table)

        headers = [
            "Method",
            "KNN Risk",
            "ROC AUC",
            "Avg Precision",
            "Classification Risk"
        ]

        for i, h in enumerate(headers):
            run = table.rows[0].cells[i].paragraphs[0].add_run(h)
            run.bold = True

        for method, result in results_dict.items():

            knn = result["MembershipKnnProbabilities"]
            clf = result["MembershipClassification"]

            row = table.add_row().cells
            row[0].text = str(method)
            row[1].text = f"{float(knn['risk_score']):.4f}"
            row[2].text = f"{float(knn['roc_auc_score']):.4f}"
            row[3].text = f"{float(knn['average_precision_score']):.4f}"
            row[4].text = f"{float(clf['risk_score']):.4f}"

    doc.save(output_path)

# **All Code, Models and Pictures dowloads**

# **All Settings**

In [ ]:
if os.getcwd().endswith(f"{base_dir}"):
  os.chdir("..")

In [ ]:
charts_path = f"./{base_dir}/Charts/"

if not os.path.exists(charts_path):
  os.makedirs(charts_path)

#!rm -rf charts_path

if not os.path.exists(f"./{base_dir}/Saved_models"):
    os.mkdir(f"./{base_dir}/Saved_models")
    !rm -rf "f./{base_dir}/Saved_models/rf_dump.model"

# **Data Reading**

In [ ]:
df = pd.read_csv(local_path, sep = ';')

df.head(15)

,tr,mese,annrif,progind,sesso,eta10,staciv4,pnasc,reg,rip,istr4,condogg,posiz4,ateco3,cond4,ORARIO,RAPP,CODPROF7,progvia,TIPOVGG,DEST_RE,DEST_PR,DEST_IE,GGINIZ,MMINIZ,AAINIZ,DURATA,MOTVAC,MEZZO,ALLOG,MOTLAV,ORGALL,ORGTRA,ALPART,IORGALL,IORGTRA,TIPOMARE,TIPOCROC,TIPOMONT,TIPOCITTA,TIPOCAMP,TIPOALTRO,W,COEV,TRIM,npart,tipo,ESPE_CO,ESPE_GIO,piattall1,piattall2,piattall3
0,3,8,2019,3,2,6,2,1,80,2,3,1.0,4.0,3.0,1.0,NaN,NaN,4.0,1,V0819,8,36.0,1,14,8,2019,3,1.0,5,14,NaN,3,3,1,NaN,NaN,2.0,2.0,1.0,2.0,2.0,2.0,5225.266670,15675.800010,3,2,7.0,115,38,NaN,NaN,NaN
1,3,5,2019,41,2,5,4,1,100,3,3,1.0,1.0,2.0,1.0,1.0,2.0,3.0,1,V0519,134,NaN,2,11,5,2019,3,1.0,1,1,NaN,2,2,2,2.0,1.0,1.0,2.0,2.0,1.0,1.0,2.0,3448.723758,10346.171273,2,1,1.0,583,194,NaN,NaN,NaN
2,3,4,2019,42,1,6,2,1,50,2,4,1.0,1.0,3.0,1.0,1.0,2.0,2.0,1,V0419,12,56.0,1,23,4,2019,3,1.0,6,1,NaN,1,2,1,3.0,3.0,2.0,2.0,2.0,1.0,2.0,2.0,6656.501991,19969.505974,2,2,1.0,484,161,NaN,NaN,NaN
3,3,6,2019,74,1,2,1,1,30,1,2,3.0,NaN,NaN,3.0,NaN,NaN,NaN,1,V0619,128,NaN,2,14,6,2019,14,1.0,1,3,NaN,1,1,2,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,7068.131027,21204.393082,2,1,4.0,1473,105,2.0,2.0,2.0
4,3,8,2019,79,1,1,1,1,80,2,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,V0819,11,42.0,1,4,8,2019,2,1.0,5,18,NaN,2,3,1,2.0,NaN,2.0,2.0,2.0,1.0,2.0,2.0,9429.311751,28287.935253,3,4,7.0,133,67,NaN,NaN,NaN
5,3,9,2019,82,2,6,2,1,150,4,3,3.0,NaN,NaN,3.0,NaN,NaN,NaN,1,V0919,3,15.0,1,25,8,2019,13,2.0,2,14,NaN,3,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5230.068875,15690.206626,3,2,NaN,204,16,NaN,NaN,NaN
6,3,11,2019,89,1,4,2,1,120,3,3,1.0,1.0,3.0,1.0,1.0,2.0,2.0,1,V1119,13,68.0,1,22,11,2019,2,1.0,5,1,NaN,2,3,1,1.0,NaN,2.0,2.0,2.0,1.0,2.0,2.0,7813.815594,23441.446783,4,4,5.0,88,44,1.0,2.0,2.0
7,3,6,2019,98,1,6,2,1,60,2,3,1.0,1.0,2.0,1.0,1.0,2.0,3.0,1,V0619,6,31.0,1,22,6,2019,8,1.0,5,14,NaN,3,3,1,NaN,NaN,1.0,2.0,2.0,1.0,2.0,2.0,2733.008790,8199.026371,2,2,2.0,73,9,NaN,NaN,NaN
8,3,9,2019,125,2,3,1,1,60,2,2,1.0,4.0,3.0,1.0,NaN,NaN,4.0,1,V0919,5,27.0,1,10,9,2019,1,1.0,5,14,NaN,3,3,2,NaN,NaN,1.0,2.0,2.0,2.0,2.0,2.0,2207.107531,6621.322593,3,1,7.0,44,44,NaN,NaN,NaN
9,3,7,2019,128,2,6,2,1,30,1,2,1.0,1.0,3.0,1.0,1.0,2.0,4.0,1,V0719,11,41.0,1,20,6,2019,14,1.0,9,1,NaN,1,3,1,2.0,NaN,1.0,2.0,2.0,1.0,2.0,2.0,6585.346374,19756.039122,3,2,7.0,650,46,NaN,NaN,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22404 entries, 0 to 22403
Data columns (total 52 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   tr         22404 non-null  int64  
 1   mese       22404 non-null  int64  
 2   annrif     22404 non-null  int64  
 3   progind    22404 non-null  int64  
 4   sesso      22404 non-null  int64  
 5   eta10      22404 non-null  int64  
 6   staciv4    22404 non-null  int64  
 7   pnasc      22404 non-null  int64  
 8   reg        22404 non-null  int64  
 9   rip        22404 non-null  int64  
 10  istr4      22404 non-null  int64  
 11  condogg    19532 non-null  float64
 12  posiz4     17099 non-null  float64
 13  ateco3     17099 non-null  float64
 14  cond4      19532 non-null  float64
 15  ORARIO     9465 non-null   float64
 16  RAPP       9465 non-null   float64
 17  CODPROF7   17099 non-null  float64
 18  progvia    22404 non-null  int64  
 19  TIPOVGG    22404 non-null  object 
 20  DEST_R

In [ ]:
df = df.drop('MOTLAV', axis=1)

In [ ]:
df = df.drop('TIPOVGG', axis=1)

# **Create and configure metadata**

In [ ]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=df)

# **Null Values Management**

In [ ]:
df = df.dropna(subset=["MOTVAC"])

In [ ]:
len(df)

20701

# **Datasets Limitations**

In [ ]:
if n_dp_limit is None:
  n_dp_limit = len(df)
if n_dcr_limit is None:
  n_dcr_limit = len(df)
if n_mia_limit is None:
  n_mia_limit = len(df)

In [ ]:
print("Number of null values per column:")
print(df.isnull().sum())

print("\nTotal number of null values in the dataset:")
print(df.isnull().sum().sum())

Number of null values per column:
tr               0
mese             0
annrif           0
progind          0
sesso            0
eta10            0
staciv4          0
pnasc            0
reg              0
rip              0
istr4            0
condogg       2872
posiz4        5241
ateco3        5241
cond4         2872
ORARIO       12266
RAPP         12266
CODPROF7      5241
progvia          0
DEST_RE          0
DEST_PR       3451
DEST_IE          0
GGINIZ           0
MMINIZ           0
AAINIZ           0
DURATA           0
MOTVAC           0
MEZZO            0
ALLOG            0
ORGALL           0
ORGTRA           0
ALPART           0
IORGALL       9790
IORGTRA      15157
TIPOMARE      5623
TIPOCROC      5623
TIPOMONT      5623
TIPOCITTA     5623
TIPOCAMP      5623
TIPOALTRO     5623
W                0
COEV             0
TRIM             0
npart            0
tipo          5623
ESPE_CO          0
ESPE_GIO         0
piattall1    17690
piattall2    17690
piattall3    17690
dtype: int64

To

In [ ]:
for c in df.columns:
    if df[c].dtype == 'object':
            df[c] = df[c].fillna("Unknown")
    else:
            df[c] = df[c].fillna(df[c].max()+1)

In [ ]:
print("Number of null values per column:")
print(df.isnull().sum())

print("\nTotal number of null values in the dataset:")
print(df.isnull().sum().sum())

Number of null values per column:
tr           0
mese         0
annrif       0
progind      0
sesso        0
eta10        0
staciv4      0
pnasc        0
reg          0
rip          0
istr4        0
condogg      0
posiz4       0
ateco3       0
cond4        0
ORARIO       0
RAPP         0
CODPROF7     0
progvia      0
DEST_RE      0
DEST_PR      0
DEST_IE      0
GGINIZ       0
MMINIZ       0
AAINIZ       0
DURATA       0
MOTVAC       0
MEZZO        0
ALLOG        0
ORGALL       0
ORGTRA       0
ALPART       0
IORGALL      0
IORGTRA      0
TIPOMARE     0
TIPOCROC     0
TIPOMONT     0
TIPOCITTA    0
TIPOCAMP     0
TIPOALTRO    0
W            0
COEV         0
TRIM         0
npart        0
tipo         0
ESPE_CO      0
ESPE_GIO     0
piattall1    0
piattall2    0
piattall3    0
dtype: int64

Total number of null values in the dataset:
0


# **Features Selection**

In [ ]:
data_x = df[features]
data_y = df[target]

# **Traditional Methods for generating synthetic data**

In [ ]:
if training is True and SMOTE is True:
  start_time = timeit.default_timer()
  # Compute the class distribution before SMOTE
  class_distribution = Counter(data_y)
  print("Distribution before SMOTE:", class_distribution)

  # Define the total size of the data after generation (e.g., doubling the dataset)
  total_samples = sum(class_distribution.values()) * 1  # Adjustable according to your needs

  # Compute the number of samples to generate per class while maintaining proportions
  sampling_strategy = {cls: int((count / sum(class_distribution.values())) * total_samples +1)
                      for cls, count in class_distribution.items()}

  # Initialize SMOTE with the customized strategy
  smote = SMOTE(sampling_strategy=sampling_strategy, random_state=seed)

  # Generate new synthetic data
  X_resampled, y_resampled = smote.fit_resample(data_x, data_y)

  # Check the class distribution after SMOTE
  print("Distribution after SMOTE:", Counter(y_resampled))

  # Combine X and y into a single DataFrame
  columns = list(data_x.columns) + ['MOTVAC']  # Assuming data_x is a DataFrame
  synthetic_data_SMOTE = pd.DataFrame(data=X_resampled, columns=data_x.columns)
  synthetic_data_SMOTE['MOTVAC'] = y_resampled

  synthetic_data_SMOTE = synthetic_data_SMOTE.sample(n=len(df), random_state=seed).reset_index(drop=True)
  print("SMOTE Forest Elapsed Time: ", timeit.default_timer() - start_time)

# **Synthetic Data Generation by Generative Models**

# **Random Forest**

In [ ]:
if training is True and RF is True:
  start_time = timeit.default_timer()
  # Create an empty DataFrame with the same shape as df to store synthetic data
  synthetic_data_rf = pd.DataFrame(np.nan, index=df.index, columns=df.columns)

  # Create a copy of the DataFrame to avoid modifying the original DataFrame during encoding
  df_encoded = df.copy()

  # Apply Target Encoding to all categorical columns before the loop
  for cat_col in df_encoded.select_dtypes(include=['object']).columns:
      encoder = ce.TargetEncoder(cols=[cat_col])
      # Use the entire dataset for fitting and transforming to ensure consistency
      df_encoded[cat_col] = encoder.fit_transform(df_encoded[cat_col], df_encoded[target])

  for col in df_encoded.columns:  # Iterate over columns of the encoded DataFrame
      print(f"Generating column: {col}...")
      # Separate columns based on their type
      if df_encoded[col].dtype == 'object':  # Categorical column
          model = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs = -1)
          # Define X and y
          X = df_encoded.drop(columns=[col])  # Use the encoded DataFrame for X
          y = df_encoded[col]  # Use the encoded DataFrame for y
      else:  # Numeric column
          model = RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs = -1)
          # Define X and y for a numeric column
          X = df_encoded.drop(columns=[col])  # Use the encoded DataFrame for X
          y = df_encoded[col]  # Use the encoded DataFrame for y

      # Split the data into train/test
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

      # Train the model
      model.fit(X_train, y_train)

      # Make predictions on the entire encoded dataset
      synthetic_data_rf[col] = model.predict(df_encoded.drop(columns=[col]))

  # Apply the function to synthetic_data_rf
  synthetic_data_rf = match_datatypes(df, synthetic_data_rf)

  print("Random Forest Elapsed Time: ", timeit.default_timer() - start_time)

# **XGBoost**

In [ ]:
if training is True and XGB is True:
  start_time = timeit.default_timer()
  # Create an empty DataFrame with the same shape as df to store synthetic data
  synthetic_data_XGB = pd.DataFrame(np.nan, index=df.index, columns=df.columns)

  # Create a copy of the DataFrame to avoid modifying the original DataFrame during encoding
  df_encoded = df.copy()

  # Apply Target Encoding to all categorical columns before the loop
  for cat_col in df_encoded.select_dtypes(include=['object']).columns:
      encoder = ce.TargetEncoder(cols=[cat_col])
      # Use the entire dataset for fitting and transforming to ensure consistency
      df_encoded[cat_col] = encoder.fit_transform(df_encoded[cat_col], df_encoded[target])

  for col in df_encoded.columns:  # Iterate over columns of the encoded DataFrame
      print(f"Generating column: {col}...")
      # Separate columns based on their type
      if df_encoded[col].dtype == 'object':  # Categorical column
          model = XGBClassifier(n_estimators=100, random_state=seed, use_label_encoder=False, eval_metric='mlogloss', enable_categorical=True)
          # Define X and y
          X = df_encoded.drop(columns=[col])  # Use the encoded DataFrame for X
          y = df_encoded[col]  # Use the encoded DataFrame for y
      else:  # Numeric column
          model = XGBRegressor(n_estimators=100, random_state=seed)
          # Define X and y for a numeric column
          X = df_encoded.drop(columns=[col])  # Use the encoded DataFrame for X
          y = df_encoded[col]  # Use the encoded DataFrame for y

      # Split the data into train/test
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

      # Train the model
      model.fit(X_train, y_train)

      # Make predictions on the entire encoded dataset
      synthetic_data_XGB[col] = model.predict(df_encoded.drop(columns=[col]))

  # Apply the function to synthetic_data_XGB
  synthetic_data_XGB = match_datatypes(df, synthetic_data_XGB)

  print("XGBoost Elapsed Time: ", timeit.default_timer() - start_time)

# **VAEs (Variational Autoencoders)**

In [ ]:
if training is True and VAE is True:
  start_time = timeit.default_timer()

  # Initialize and train the model
  synthesizer = TVAESynthesizer(metadata, batch_size=8192)
  synthesizer.fit(df)

  # Generate synthetic data
  synthetic_data_VAE = synthesizer.sample(num_rows=len(df))
  print("VAE Elapsed Time: ", timeit.default_timer() - start_time)

# **CT-GANS (Conditional Tabular GANs)**

In [ ]:
if training is True and CTGAN is True:
  start_time = timeit.default_timer()

  # Initialize and train the model
  synthesizer = CTGANSynthesizer(metadata)
  synthesizer.fit(df)

  # Generate synthetic data
  synthetic_data_CTGAN = synthesizer.sample(num_rows=len(df))

  print("CT-Gans Elapsed Time: ", timeit.default_timer() - start_time)

# **DP-CTGAN (Differential Privacy - Conditional Tabular GAN)**

In [ ]:
if training is True and DPCTGAN is True:
    start_time = timeit.default_timer()

    synthesizer = Synthesizer.create(
        "dpctgan",
        epsilon=dpctgan_epsilon,   # moderate privacy
        delta=dpctgan_delta,       # High privacy guarantee
        epochs=dpctgan_epochs,
        batch_size=dpctgan_batch_size,
        verbose=True
    )

    synthesizer.fit(
        df,
        categorical_columns=df.select_dtypes(include=['object', 'category']).columns.tolist()
        ,
        continuous_columns=df.select_dtypes(include=['number']).columns.tolist()
        ,
        preprocessor_eps=0.5
    )

    synthetic_data_DPCTGAN = synthesizer.sample(len(df))

    print("DP-CTGAN Elapsed Time:", timeit.default_timer() - start_time)

# **Export of Generated Synthetic Data**

In [ ]:
if training is True:
  if SMOTE is True:
    synthetic_data_SMOTE.to_excel(local_results_path + "synthetic_data_SMOTE.xlsx", index=False, engine='openpyxl')
  if RF is True:
    synthetic_data_rf.to_excel(local_results_path + "synthetic_data_RF.xlsx", index=False, engine='openpyxl')
  if XGB is True:
    synthetic_data_XGB.to_excel(local_results_path + "synthetic_data_XGB.xlsx", index=False, engine='openpyxl')
  if VAE is True:
    synthetic_data_VAE.to_excel(local_results_path + "synthetic_data_VAE.xlsx", index=False, engine='openpyxl')
  if CTGAN is True:
    synthetic_data_CTGAN.to_excel(local_results_path + "synthetic_data_CTGAN.xlsx", index=False, engine='openpyxl')
  if DPCTGAN is True:
    synthetic_data_DPCTGAN.to_excel(local_results_path + "synthetic_data_DPCTGAN.xlsx", index=False, engine='openpyxl')

In [ ]:
# Read each synthetic dataset from its Excel file
if training is False:
  if SMOTE is True:
    synthetic_data_SMOTE = pd.read_excel(local_results_path + "synthetic_data_SMOTE.xlsx", engine='openpyxl')
    synthetic_data_SMOTE = match_datatypes(df, synthetic_data_SMOTE)
  if RF is True:
    synthetic_data_rf = pd.read_excel(local_results_path + "synthetic_data_RF.xlsx", engine='openpyxl')
    synthetic_data_rf = match_datatypes(df, synthetic_data_rf)
  if XGB is True:
    synthetic_data_XGB = pd.read_excel(local_results_path + "synthetic_data_XGB.xlsx", engine='openpyxl')
    synthetic_data_XGB = match_datatypes(df, synthetic_data_XGB)
  if VAE is True:
    synthetic_data_VAE = pd.read_excel(local_results_path + "synthetic_data_VAE.xlsx", engine='openpyxl')
    synthetic_data_VAE = match_datatypes(df, synthetic_data_VAE)
  if CTGAN is True:
    synthetic_data_CTGAN = pd.read_excel(local_results_path + "synthetic_data_CTGAN.xlsx", engine='openpyxl')
    synthetic_data_CTGAN = match_datatypes(df, synthetic_data_CTGAN)
  if DPCTGAN is True:
    synthetic_data_DPCTGAN = pd.read_excel(local_results_path + "synthetic_data_DPCTGAN.xlsx", engine='openpyxl')
    synthetic_data_DPCTGAN = match_datatypes(df, synthetic_data_DPCTGAN)

  print("Synthetic datasets loaded successfully.")
  print("Data type matching complete for all loaded synthetic dataframes.")

Synthetic datasets loaded successfully.
Data type matching complete for all loaded synthetic dataframes.


In [ ]:
synthetic_datasets = {}

if SMOTE is True:
    synthetic_datasets["SMOTE"] = synthetic_data_SMOTE
if RF is True:
    synthetic_datasets["RF"] = synthetic_data_rf
if XGB is True:
    synthetic_datasets["XGB"] = synthetic_data_XGB
if VAE is True:
    synthetic_datasets["VAE"] = synthetic_data_VAE
if CTGAN is True:
    synthetic_datasets["CTGAN"] = synthetic_data_CTGAN
if DPCTGAN is True:
    synthetic_datasets["DPCTGAN"] = synthetic_data_DPCTGAN

# **Utility Metrics**

## **Machine Learning Metrics**

# **Privacy Metrics**

## **Disclosure Protection**

**Disclosure Protection (DP)** is a metric used to evaluate the privacy guarantees provided by a synthetic dataset. Its purpose is to measure the risk that an attacker could infer sensitive information about individuals by exploiting the synthetic data together with some external knowledge.

More specifically, the metric assesses the likelihood of **attribute disclosure**, which occurs when an adversary is able to correctly infer sensitive attributes of a record by leveraging known characteristics of the same individual.

The evaluation framework assumes the presence of an attacker who possesses **partial background knowledge** about individuals contained in the original dataset. Let **K** denote the set of **known attributes** (also referred to as quasi-identifiers), and **S** the set of **sensitive attributes** that the attacker aims to infer.

The attack scenario is defined as follows: if an attacker knows the values of the attributes in **K** for an individual (i) in the original dataset, they search the synthetic dataset for records that share the same or similar values for these attributes. Based on these matches, the attacker attempts to **predict the values of the sensitive attributes (S)** for that individual.

Disclosure Protection quantifies how accurately such predictions can be made. If the synthetic data closely reproduces the relationships between quasi-identifiers and sensitive variables present in the original data, an attacker may be able to infer sensitive information with high confidence, resulting in **low privacy protection**. Conversely, if these relationships are sufficiently altered or obscured in the synthetic dataset, the attack becomes less effective, leading to **higher disclosure protection**.

Therefore, DP captures the **trade-off between data utility and privacy**: synthetic datasets that strongly preserve the statistical structure of the original data may provide higher analytical utility but can also increase the risk of attribute disclosure. A robust synthetic data generation method should aim to maintain useful statistical properties while minimizing the risk that sensitive attributes can be inferred.

In practice, Disclosure Protection is often computed by simulating the attacker’s inference process and measuring the **success rate of predicting sensitive attributes**, given the known attributes. Lower prediction accuracy corresponds to stronger privacy protection.



In [ ]:
# hyphotesis
sensitive_columns = [target]
#known_column_names = ["sesso", "eta10", "staciv4", "pnasc", "reg"]
known_column_names = ["staciv4", "pnasc", "reg"]
#known_column_names = features

In [ ]:
if dp_privacy is True:
    start_global_time = timeit.default_timer()
    dp_results_all = {}

    for name, synth_df in synthetic_datasets.items():
        start_time = timeit.default_timer()

        dp_score = DisclosureProtection.compute(
            df[0:n_dp_limit],
            synth_df[0:n_dp_limit],
            known_column_names,
            sensitive_columns
        )

        dp_results_all[name] = dp_score

        print(f"Disclosure Protection Original DF - Synthetic Data by {name}: {dp_score}")
        print(f"{name} DP Elapsed Time: {timeit.default_timer() - start_time}")

    print("\nAll DP calculations completed.")
    print(dp_results_all)

    save_results_to_word(
        dp_results_all,
        local_results_path + "dp_results_table.docx",
        "Disclosure Protection Results",
        "DP Score"
    )

    print("\nDP Elapsed Time:", timeit.default_timer() - start_global_time)

/usr/local/lib/python3.12/dist-packages/sdmetrics/single_table/privacy/disclosure_protection.py:214: UserWarning: Data exceeds 10000 rows, perfomance may be slow. Consider using the `DisclosureProtectionEstimate` for faster computation.
  warnings.warn(


Disclosure Protection Original DF - Synthetic Data by SMOTE: 0.5136557877647075
SMOTE DP Elapsed Time: 50.057999849


/usr/local/lib/python3.12/dist-packages/sdmetrics/single_table/privacy/disclosure_protection.py:214: UserWarning: Data exceeds 10000 rows, perfomance may be slow. Consider using the `DisclosureProtectionEstimate` for faster computation.
  warnings.warn(


Disclosure Protection Original DF - Synthetic Data by RF: 0.5047656284772354
RF DP Elapsed Time: 47.58969985900001


/usr/local/lib/python3.12/dist-packages/sdmetrics/single_table/privacy/disclosure_protection.py:214: UserWarning: Data exceeds 10000 rows, perfomance may be slow. Consider using the `DisclosureProtectionEstimate` for faster computation.
  warnings.warn(


...

Disclosure Protection Original DF - Synthetic Data by VAE: 0.5366971317059038
VAE DP Elapsed Time: 47.38280726400001


/usr/local/lib/python3.12/dist-packages/sdmetrics/single_table/privacy/disclosure_protection.py:214: UserWarning: Data exceeds 10000 rows, perfomance may be slow. Consider using the `DisclosureProtectionEstimate` for faster computation.
  warnings.warn(


Disclosure Protection Original DF - Synthetic Data by CTGAN: 0.5642266261791313
CTGAN DP Elapsed Time: 47.29207221699994


/usr/local/lib/python3.12/dist-packages/sdmetrics/single_table/privacy/disclosure_protection.py:214: UserWarning: Data exceeds 10000 rows, perfomance may be slow. Consider using the `DisclosureProtectionEstimate` for faster computation.
  warnings.warn(


Disclosure Protection Original DF - Synthetic Data by DPCTGAN: 1
DPCTGAN DP Elapsed Time: 46.35977596000009

All DP calculations completed.
{'SMOTE': 0.5136557877647075, 'RF': 0.5047656284772354, 'XGB': 1, 'VAE': 0.5366971317059038, 'CTGAN': 0.5642266261791313, 'DPCTGAN': 1}

DP Elapsed Time: 286.198672763


## **DCR Baseline Protection**

DCR (Distance to Closest Record) baseline protection is a privacy risk metric used to check whether synthetic data is too similar to real data. The idea is simple:

* If synthetic records are as “far” from real records as random data would be, then privacy risk is low.

Let’s break it down clearly, also using your description.

### 🔹 Step 1 — Compute distances between synthetic and real data

For **each row in the synthetic dataset**, compute its distance to all rows in the real dataset.

Distance depends on variable type:

* **Numerical variables** → Euclidean distance

$$
d(x, y) = \sqrt{\sum_i (x_i - y_i)^2}
$$

* **Categorical variables** → Hamming-like distance

  * 0 if values are the same
  * 1 if values are different

For mixed data types, a combined metric such as **Gower distance** is often used.

### 🔹 Step 2 — Find the closest real record

For each synthetic record ( s ), compute its distance to every real record ( r ), and keep the minimum:

$$
DCR_s = \min_{r \in Real} d(s, r)
$$

So each synthetic row gets **one value**:
👉 the distance to its **closest real record**.

### 🔹 Step 3 — Aggregate using the median

To obtain a dataset-level metric, compute:

$$
\text{Median}(DCR_{synthetic})
$$

The **median** is preferred over the minimum because it is more robust to outliers and gives a stable global privacy signal.



### 🔹 Step 4 — Build a random baseline

Now create a **random dataset** with:

* Same number of rows
* Same variables
* Similar marginal distributions

But **not generated from the real dataset**.

Repeat the same process:

$$
DCR_{random} = \min_{r \in Real} d(x_{random}, r)
$$

$$
\text{Median}(DCR_{random})
$$


### 🔹 Step 5 — Compute the DCR ratio

$$
\textbf{DCR Ratio} =
\frac{\text{Median}(DCR_{synthetic})}
{\text{Median}(DCR_{random})}
$$

This ratio tells us how close synthetic data is to real data compared to random noise.



## 🔍 Interpretation

| DCR Ratio | Meaning                                                   | Privacy Risk                           |
| --------- | --------------------------------------------------------- | -------------------------------------- |
| **≈ 1**   | Synthetic data is as far from real data as random data is | ✅ Low risk                             |
| **< 1**   | Synthetic data is closer to real data than random data    | ⚠️ Possible memorization               |
| **≪ 1**   | Many synthetic records are very close to real ones        | 🚨 High risk                           |
| **> 1**   | Synthetic data is farther than random                     | 🔒 Very safe, but possibly low utility |

---

## 🧠 Intuition

* If a model **memorizes** real individuals → distances become very small → **low DCR ratio**
* If synthetic data behaves like **random noise** → distances similar to random → **ratio ≈ 1**
* If synthetic data is **too noisy** → distances larger than random → **ratio > 1**

---

## 🎯 What DCR Helps Detect

DCR is useful for identifying:

* Record memorization
* Near-duplicates of real individuals
* Privacy leakage in generative models
* Membership inference risk

It is widely used to evaluate privacy in **GANs, VAEs, diffusion models**, and other synthetic data generators.

---

### ✅ One-line summary

**DCR baseline protection checks whether synthetic data points are closer to real data than random data would be — if not, privacy is considered preserved.**


A ratio of 0.218 suggests:

🚨 Many synthetic records lie very near real individuals
🚨 Possible memorization or overfitting
🚨 Higher risk of:

Record linkage

Membership inference attacks

Identity disclosure

The generator may be reproducing training data patterns too faithfully.

In [ ]:
if dcr_privacy is True:
    start_global_time = timeit.default_timer()
    dcr_results_all = {}

    for name, synth_df in synthetic_datasets.items():
        start_time = timeit.default_timer()

        dcr_baseline_score = DCRBaselineProtection.compute(
            real_data=df[0:n_dcr_limit],
            synthetic_data=synth_df[0:n_dcr_limit],
            metadata=metadata.to_dict()
        )

        dcr_results_all[name] = dcr_baseline_score

        print(f"DCR Baseline Protection for {name}: {dcr_baseline_score}")
        print(f"{name} DCR Elapsed Time: {timeit.default_timer() - start_time}")

    print("\nAll DCR calculations completed.")
    print(dcr_results_all)

    save_results_to_word(
        dcr_results_all,
        local_results_path + "dcr_results_table.docx",
        "DCR Baseline Protection Results",
        "DCR Score"
    )

    print("\nDCR Elapsed Time:", timeit.default_timer() - start_global_time)

DCR Baseline Protection for SMOTE: 0.27909383660032655
SMOTE DCR Elapsed Time: 92.36024156599979
DCR Baseline Protection for RF: 0.24657590040836516
RF DCR Elapsed Time: 86.99098056699995
DCR Baseline Protection for XGB: 0.7231037589651126
XGB DCR Elapsed Time: 86.79692829700002
DCR Baseline Protection for VAE: 0.26143021375603714
VAE DCR Elapsed Time: 86.23637719199996
DCR Baseline Protection for CTGAN: 0.3377016001534772
CTGAN DCR Elapsed Time: 86.88534082800015
DCR Baseline Protection for DPCTGAN: 0.9763333803829277
DPCTGAN DCR Elapsed Time: 86.96949463800001

All DCR calculations completed.
{'SMOTE': 0.27909383660032655, 'RF': 0.24657590040836516, 'XGB': 0.7231037589651126, 'VAE': 0.26143021375603714, 'CTGAN': 0.3377016001534772, 'DPCTGAN': 0.9763333803829277}

DCR Elapsed Time: 526.2728444439999


## **Membership Inference Attack (MIA)**

In [ ]:
def MIA_attack(X_real, y_real, X_synth, y_synth, X_test, y_test):
    results = {}

    # 1️⃣ Membership KNN Probabilities
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_real, y_real)

    # Probability
    prob_members = knn.predict_proba(X_real)[:, 1]  # members
    prob_non_members = knn.predict_proba(X_test)[:, 1]  # non-members
    prob_synth = knn.predict_proba(X_synth)[:, 1]  # synthetic

    # Labels 1 for members, 0 for non-membres
    y_true = np.concatenate([np.ones_like(prob_members), np.zeros_like(prob_non_members)])
    y_scores = np.concatenate([prob_members, prob_non_members])

    mia_knn_auc = roc_auc_score(y_true, y_scores)
    mia_knn_ap = average_precision_score(y_true, y_scores)

    results["MembershipKnnProbabilities"] = {
        "risk_score": np.mean(prob_synth),
        "roc_auc_score": mia_knn_auc,
        "average_precision_score": mia_knn_ap,
    }

    print("=== MembershipKnnProbabilities ===")
    print("Risque (risk_score) :", results["MembershipKnnProbabilities"]["risk_score"])
    print("ROC AUC score :", results["MembershipKnnProbabilities"]["roc_auc_score"])
    print("Average precision score :", results["MembershipKnnProbabilities"]["average_precision_score"])

    # 2️⃣ Membership Classification
    # Binary dataset (member vs non-member)
    X_attack = np.vstack([X_real, X_test])
    y_attack = np.hstack([np.ones(len(X_real)), np.zeros(len(X_test))])

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_attack, y_attack)

    # Scores
    scores_members = clf.predict_proba(X_real)[:, 1]
    scores_non_members = clf.predict_proba(X_test)[:, 1]

    member_auc = roc_auc_score(np.ones(len(scores_members)), scores_members)
    non_member_auc = roc_auc_score(np.zeros(len(scores_non_members)), scores_non_members)

    results["MembershipClassification"] = {
        "risk_score": clf.predict_proba(X_synth)[:, 1].mean(),
        "member_roc_auc_score": member_auc,
        "non_member_roc_auc_score": non_member_auc,
    }

    print("\n=== MembershipClassification ===")
    print("Risque (risk_score) :", results["MembershipClassification"]["risk_score"])
    print("Member ROC AUC score :", results["MembershipClassification"]["member_roc_auc_score"])
    print("Non-member ROC AUC score :", results["MembershipClassification"]["non_member_roc_auc_score"])

    return results

In [ ]:
if mia_privacy is True:
    start_global_time = timeit.default_timer()

    X = df.drop(columns=[target])[0:n_mia_limit]
    y = df[target][0:n_mia_limit]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    X_train_mia = X_train.to_numpy()
    y_train_mia = y_train.to_numpy()
    X_test_mia = X_test.to_numpy()
    y_test_mia = y_test.to_numpy()

    mia_results_all = {}

    for name, synth_df in synthetic_datasets.items():
        synth_df = synth_df[0:n_mia_limit]
        print(f"\n\nRunning MIA attack for {name}...")

        X_synth_mia_current = synth_df.drop(columns=[target]).to_numpy()
        y_synth_mia_current = synth_df[target].to_numpy()

        mia_results_all[name] = MIA_attack(
            X_train_mia, y_train_mia,
            X_synth_mia_current, y_synth_mia_current,
            X_test_mia, y_test_mia
        )

    print("\nAll MIA attacks completed.")
    print(mia_results_all)

    save_results_to_word(
        mia_results_all,
        local_results_path + "mia_results_table.docx",
        "Membership Inference Attack Results",
        "MIA Score"
    )

    print("\nMIA Elapsed Time:", timeit.default_timer() - start_global_time)



Running MIA attack for SMOTE...
=== MembershipKnnProbabilities ===
Risque (risk_score) : 0.25545625815178014
ROC AUC score : 0.49530949781086275
Average precision score : 0.8005830098014569


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



=== MembershipClassification ===
Risque (risk_score) : 0.7999660812818606
Member ROC AUC score : nan
Non-member ROC AUC score : nan


Running MIA attack for RF...
=== MembershipKnnProbabilities ===
Risque (risk_score) : 0.2587894304622965
ROC AUC score : 0.49530949781086275
Average precision score : 0.8005830098014569


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



=== MembershipClassification ===
Risque (risk_score) : 0.8000150145375962
Member ROC AUC score : nan
Non-member ROC AUC score : nan


Running MIA attack for XGB...
=== MembershipKnnProbabilities ===
Risque (risk_score) : 0.2628858509250761
ROC AUC score : 0.49530949781086275
Average precision score : 0.8005830098014569


...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



=== MembershipClassification ===
Risque (risk_score) : 0.7994854585228742
Member ROC AUC score : nan
Non-member ROC AUC score : nan


Running MIA attack for CTGAN...
=== MembershipKnnProbabilities ===
Risque (risk_score) : 0.24179508236317085
ROC AUC score : 0.49530949781086275
Average precision score : 0.8005830098014569


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



=== MembershipClassification ===
Risque (risk_score) : 0.8022592761813523
Member ROC AUC score : nan
Non-member ROC AUC score : nan


Running MIA attack for DPCTGAN...
=== MembershipKnnProbabilities ===
Risque (risk_score) : 0.3022462682962176
ROC AUC score : 0.49530949781086275
Average precision score : 0.8005830098014569

=== MembershipClassification ===
Risque (risk_score) : 0.8010292108784761
Member ROC AUC score : nan
Non-member ROC AUC score : nan

All MIA attacks completed.
{'SMOTE': {'MembershipKnnProbabilities': {'risk_score': np.float64(0.25545625815178014), 'roc_auc_score': np.float64(0.49530949781086275), 'average_precision_score': np.float64(0.8005830098014569)}, 'MembershipClassification': {'risk_score': np.float64(0.7999660812818606), 'member_roc_auc_score': nan, 'non_member_roc_auc_score': nan}}, 'RF': {'MembershipKnnProbabilities': {'risk_score': np.float64(0.2587894304622965), 'roc_auc_score': np.float64(0.49530949781086275), 'average_precision_score': np.float64(0.8

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


# **Github Upload**

/content/AIML4OS_WP_13_synthetic_data_modelling_task_team
Already up to date.
[main cb9ac88] New Results and Models in Death Causes Classification
 3 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 Results/Synthetic_Data/dcr_results_table.docx
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 36.77 KiB | 12.26 MiB/s, done.
Total 7 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 3 local objects.
To https://github.com/istat-methodology/AIML4OS_WP_13_synthetic_data_modelling_task_team.git
   46be1d2..cb9ac88  main -> main


In [ ]:
print("Global Elapsed Time: ", timeit.default_timer() - start_global_time)

Global Elapsed Time:  75.65165944599994
